# THERMA SEARCH AI · 04 · HIT-UAV YOLO11s exploratory baseline

**Colab GPU seçin → Çalışma zamanı → Tümünü çalıştır.** İlk Drive bağlantısında Google izni gerekir. Bu notebook GPU üzerinde gerçek eğitimi başlatır ve gerektiğinde resmi YOLO11s pretrained ağırlığını indirir; kullanıcı bu eğitim akışını yetkilendirmiştir. Veri seti indirilmez. GPU yoksa uzun CPU eğitimi yapılmaz.

Kaynak ZIP: `/content/drive/MyDrive/ThermaSearchAI/datasets/hit_uav_v1.2.1.zip`.
Çıktılar: `/content/drive/MyDrive/ThermaSearchAI/runs/hit_uav_yolo11s/run_yolo11s_.../`.

**Bilimsel kapsam:** Bu bir exploratory baseline’dır. Önceki kullanıcı denetiminde bildirilen 32 sıfır ham anotasyonlu görüntü ve 1.880 split dışı dHash adayı risk olarak korunur; bunlar burada yeniden ölçülmüş kabul edilmez. Bağımsız saha/uçuş genellemesi iddiası yapılmaz. Özgün splitler yeniden bölünmez.

**Politika:** Sıfır ham anotasyonlu ve DontCare içeren görüntüler bütün splitlerin türetilmiş kopyalarından dışlanır. Diğer bilinen sınıfların kutuları hedef dışıdır; bu görüntüler Person açısından arka plan kabul edilir (Person etiketlerinin tamlığı varsayımı). Standart YOLO ignore bölgelerini desteklemediğinden DontCare sessizce arka plana çevrilmez. Bu politika test kapsamını da daraltır; sonuç tam özgün 579 görüntülük test skoru değildir. Her dışlama ve kaybedilen Person sayısı manifestte raporlanır. Kaynak ZIP ve etiketler değiştirilmez.

## 1 · Sınanan kodu ve eğitim bağımlılığını hazırla
Notebook 01–03 bağımlılıkları değişmez. Colab’ın CUDA torch/torchvision kurulumu kullanılır; CPU ortamında eğitim başlamaz. `ultralytics==8.3.228` sabittir.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

REPOSITORY = 'https://github.com/hilalaltunayy/therma-search-ai.git'
CODE_REF = 'feature/hit-uav-yolo11s-baseline'  # Teslimde doğrulanan yardımcı kod commit’ine sabitlenir.
local_code_root = os.environ.get('THERMA_CODE_ROOT')  # Yalnızca yerel sentetik doğrulama.
if local_code_root:
    PROJECT_ROOT = Path(local_code_root).resolve()
else:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU yok. Colab çalışma zamanı türünden GPU seçin; CPU eğitimi başlatılmadı.')
    PROJECT_ROOT = Path('/content/therma-search-ai-training')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', '--quiet', REPOSITORY, str(PROJECT_ROOT)], check=True)
    remote = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('Kod klasörünün kaynağı beklenen depo değil.')
    if subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Yerel kod değişiklikleri var; üzerine yazılmadı. Temiz Colab çalışma zamanı kullanın.')
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--quiet', '--depth', '1', 'origin', CODE_REF], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade-strategy', 'only-if-needed', '-r', str(PROJECT_ROOT/'requirements-training-colab.txt')], check=True)
sys.path.insert(0, str(PROJECT_ROOT))
from src.training.baseline import BaselineConfig, run_baseline
from IPython.display import display, Image as DisplayImage
CODE_REVISION = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Kod revizyonu:', CODE_REVISION)

## 2 · Drive ve önceden belirlenmiş ayarlar
İlk çalışma ayarları: yolo11s.pt, detect, tek sınıf Person, imgsz=640, epochs=40, batch=8, patience=10, seed=42, pretrained=True. GPU toplam belleği 6 GiB altındaysa batch en fazla 4 olur; karar kaydedilir. AMP kapalıdır (FP32); başka AMP-kontrol ağırlığı indirilmez. OOM durumunda gizli tekrar/CPU fallback yapılmaz; hata ve kısmi çıktılar korunur.

In [ ]:
if not local_code_root:
    from google.colab import drive
    drive.mount('/content/drive')
ARCHIVE_PATH = Path(os.environ.get('THERMA_TRAIN_ZIP', '/content/drive/MyDrive/ThermaSearchAI/datasets/hit_uav_v1.2.1.zip'))
OUTPUT_ROOT = Path(os.environ.get('THERMA_TRAIN_OUTPUT', '/content/drive/MyDrive/ThermaSearchAI/runs/hit_uav_yolo11s'))
WORK_ROOT = Path(os.environ.get('THERMA_TRAIN_WORK', '/content/therma-yolo-work'))
CONFIG = BaselineConfig()
print('Kaynak:', ARCHIVE_PATH)
print('Kalıcı çalışma dizinleri:', OUTPUT_ROOT)
print('Ayarlar:', CONFIG)

## 3 · Dönüştür, doğrula, eğit ve ayrı test et

Tek çağrı: kaynak JSON/görüntü bütünlüğü → ayrı yerel YOLO kopyası → normalize kutu/ters dönüşüm/hash denetimi → GPU train + validation → **validation ile seçilen best.pt** → ayrı test. Veri hatası varsa model yükleme ve eğitim başlamaz. Kopya `/content/therma-yolo-work/` altında kalır; kaynak arşivden tekrar üretilebilir. Manifest, dışlama listesi ve veri YAML’ı Drive’a kaydedilir.

Her epoch train/val metrik ve loss değerleri `ultralytics/train/results.csv` içine yazılır; best/last checkpoint’ler Drive’da tutulur. 40 epoch’a ulaşmadan patience=10 erken durdurabilir. Test eğitim sırasında kullanılmaz; confidence=.25, eşleme IoU=.50, NMS IoU=.70 ve max_det=300 önceden sabittir. mAP için ayrı Ultralytics değerlendirmesinde conf tabanı=.001 kullanılır. Native eğri optimumundan test eşiği seçilmez.

In [ ]:
TRAIN_RUN_DIR, TRAIN_SUMMARY = run_baseline(
    ARCHIVE_PATH, OUTPUT_ROOT, WORK_ROOT,
    code_revision=CODE_REVISION, config=CONFIG,
)

## 4 · Gerçek çalışma sonuçları
Aşağıdaki değerler yalnızca başarılı GPU eğitimi ve ayrı test tamamlanırsa oluşur. PR’nin sentetik testleri model başarımı değildir. Confusion matrix’te gerçek negatif nesne sayısı tanımlanmaz. FP/FN örneği yoksa ilgili görsel üretilmez.

In [ ]:
print('Durum:', TRAIN_SUMMARY['status'])
print('mAP:', json.dumps(TRAIN_SUMMARY['test_map'], ensure_ascii=False, indent=2))
print('Sabit eşikte Precision/Recall:', json.dumps(TRAIN_SUMMARY['test_fixed_threshold'], ensure_ascii=False, indent=2))
for path in [TRAIN_RUN_DIR/'training_validation_losses.png',
             TRAIN_RUN_DIR/'test_diagnostics/fixed_threshold_confusion_matrix.png',
             *sorted((TRAIN_RUN_DIR/'ultralytics/test').glob('*PR_curve.png')),
             *sorted((TRAIN_RUN_DIR/'test_diagnostics').glob('comparison_*.png'))[:2],
             *sorted((TRAIN_RUN_DIR/'test_diagnostics').glob('false_positive_*.png'))[:1],
             *sorted((TRAIN_RUN_DIR/'test_diagnostics').glob('false_negative_*.png'))[:1]]:
    if path.exists():
        display(DisplayImage(filename=str(path), width=1000))
print('En iyi model:', TRAIN_RUN_DIR/'best.pt')
print('Rapor:', TRAIN_RUN_DIR/'training_report.md')
print('Özgün splitler korunmuştur; filtrelenmiş test sonucu exploratory baseline’dır.')

## 5 · Sonraki uygulama entegrasyonu
`src.evaluation.inference.load_person_detector(best_path)` ile güvenilen modeli süreç başında bir kez yükleyin; `predict_person(model, image)` orijinal görüntü boyutları, piksel xyxy kutuları, confidence ve sınıf döndürür. Bu notebook tekrar inference eğitimi başlatmaz.

FastAPI + Next.js için [ortak inference API sözleşmesi](https://github.com/hilalaltunayy/therma-search-ai/blob/main/docs/inference_api_contract.md) hazırlanmıştır. Bu teslim frontend veya sunucu endpoint’i oluşturmaz; Claude’un frontend çalışmasına müdahale edilmez. Model checkpoint’i ve gerçek görseller GitHub’a gönderilmez.